# Phase 1: Pretraining: MLA decoder

This is the **Pretraining** part of the full `mla` notebook (`notebooks/CS781_mla_all_phases.ipynb`), extracted so it can be read on its own. It reuses the tokenizer, model classes and helpers defined in the earlier cells, so **to run it, execute the full notebook from the top** (or run the Phase 1 set-up cells first, then this part).

# CS781 Capstone: MLA decoder, Phases 1 to 7 (full notebook)

This notebook trains the **MLA decoder** from scratch (Phase 1) and then runs every downstream phase on it: Phase 2 NER, Phase 3 rhetorical roles, Phase 4 court judgment prediction, Phase 5 statute identification, Phase 6 summarisation and Phase 7 prior case retrieval. Each phase starts at a heading with its name. Run the cells **top to bottom**; later phases reuse the tokenizer, model classes and helper functions defined earlier. See the repository README for set-up (Colab GPU, Drive, data access secret) and the per-phase folders for the scores of every run.

## 1. Data Access Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [1]:
import os
from huggingface_hub import login
from google.colab import userdata

# Retrieve the token from your Colab Secrets (make sure a secret named 'HF' is saved in your notebook settings)
HF_TOKEN = userdata.get('HF')
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful on Google Colab.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful on Google Colab.


## 2. Check GPU

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

CUDA available: True
GPU count: 1


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [3]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# Updated training files list (Supreme Court, Acts, and Tribunals)
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_supreme_court.json",
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_tribunals.json"
]
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 1_500_000_000
VAL_TOKENS = 10_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 4096

# Model architecture configured to match the previous run (n_layer = 8)
N_EMBD = 768
N_LAYER = 8
N_HEAD = 12

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.6
TOP_P = 0.9

print("Configuration updated successfully.")

Configuration updated successfully.


## 4. Load Tokenizer

In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


### Tokenizer implementation (from scratch)

Byte-level BPE written by hand: word splitting, merge training, encoding and decoding. The cells below load the saved merge table with it.

In [ ]:
# ============================================================
# Byte-level BPE tokenizer, implemented from scratch
#   - pretokenize : split text into words with a regular expression, map every byte to a printable symbol
#   - train_bpe   : learn the merge table (counts word frequencies, then repeatedly merges the most frequent pair)
#   - ByteBPE     : encode / decode with a learned merge table
# ============================================================
import json, heapq, time
from collections import Counter, defaultdict
try:
    import regex as re          # needed for the Unicode classes \p{L} and \p{N} (installed on Colab)
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
except ImportError:
    import re
    PAT = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""
PAT = re.compile(PAT)

def bytes_to_unicode():
    """Map each of the 256 byte values to one printable character (so every token is a readable string)."""
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
    cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return {b: chr(c) for b, c in zip(bs, cs)}
B2U = bytes_to_unicode()
U2B = {v: k for k, v in B2U.items()}
SPECIALS = ["<|endoftext|>", "[PAD]"]
SPECIAL_SPLIT = re.compile("(" + "|".join(re.escape(s) for s in SPECIALS) + ")")

def pretokenize(text):
    return ["".join(B2U[b] for b in w.encode("utf-8")) for w in PAT.findall(text)]

def train_bpe(texts, vocab_size=32000, min_frequency=2):
    wc = Counter()
    for t in texts:
        wc.update(pretokenize(t))
    alphabet = sorted({c for w in wc for c in w})
    vocab = {s: i for i, s in enumerate(SPECIALS)}
    for c in alphabet: vocab[c] = len(vocab)
    words = [[vocab[c] for c in w] for w in wc]
    freqs = list(wc.values())
    pair_cnt = defaultdict(int); where = defaultdict(set)
    for wi, w in enumerate(words):
        for p in zip(w, w[1:]):
            pair_cnt[p] += freqs[wi]; where[p].add(wi)
    heap = [(-c, p) for p, c in pair_cnt.items()]; heapq.heapify(heap)
    id2tok = {i: s for s, i in vocab.items()}; merges = []
    while len(vocab) < vocab_size and heap:
        negc, pair = heapq.heappop(heap)
        if pair_cnt.get(pair, 0) != -negc:
            if pair_cnt.get(pair, 0) > 0: heapq.heappush(heap, (-pair_cnt[pair], pair))
            continue
        if -negc < min_frequency: break
        a, b = pair; new_tok = id2tok[a] + id2tok[b]
        new_id = len(vocab); vocab[new_tok] = new_id; id2tok[new_id] = new_tok
        merges.append((id2tok[a], id2tok[b]))
        touched = set()
        for wi in list(where[pair]):
            w, f = words[wi], freqs[wi]
            for p in zip(w, w[1:]):
                pair_cnt[p] -= f
                if p != pair: touched.add(p)
            out, i = [], 0
            while i < len(w):
                if i < len(w) - 1 and w[i] == a and w[i + 1] == b: out.append(new_id); i += 2
                else: out.append(w[i]); i += 1
            words[wi] = out
            for p in zip(out, out[1:]):
                pair_cnt[p] += f; where[p].add(wi); touched.add(p)
        pair_cnt[pair] = 0; del where[pair]
        for p in touched:
            if pair_cnt[p] > 0: heapq.heappush(heap, (-pair_cnt[p], p))
    return vocab, merges

class _Enc:                                   # small result object so callers can write  tokenizer.encode(text).ids
    def __init__(self, ids): self.ids = ids

class ByteBPE:
    def __init__(self, vocab, merges):
        self.vocab, self.id2tok = vocab, {i: t for t, i in vocab.items()}
        self.merges = [tuple(m) for m in merges]
        self.rank = {m: r for r, m in enumerate(self.merges)}; self.cache = {}
    def _parts(self, w):
        parts = list(w)
        while len(parts) > 1:
            best = min((self.rank.get(p, 1 << 30), i) for i, p in enumerate(zip(parts, parts[1:])))
            if best[0] == 1 << 30: break
            i = best[1]; parts[i:i + 2] = [parts[i] + parts[i + 1]]
        return parts
    def _bpe(self, w):
        if w not in self.cache:
            self.cache[w] = [self.vocab.get(p, 0) for p in self._parts(w)]   # a symbol never seen in training maps to id 0
        return self.cache[w]
    def get_vocab_size(self): return len(self.vocab)
    def token_to_id(self, t): return self.vocab.get(t)
    def encode_ids(self, text):
        ids = []
        for piece in SPECIAL_SPLIT.split(text):
            if piece in self.vocab and piece in SPECIALS: ids.append(self.vocab[piece])
            elif piece:
                for w in pretokenize(piece): ids += self._bpe(w)
        return ids
    def encode(self, text):
        """Like encode_ids, but also returns the character span (start, end) of every token in `text`.
        Leading and trailing spaces are trimmed from a token's span, so the span covers only visible characters."""
        ids, toks, offs = [], [], []
        pos = 0
        for piece in SPECIAL_SPLIT.split(text):
            if piece in SPECIALS:
                ids.append(self.vocab[piece]); toks.append(piece); offs.append((pos, pos + len(piece)))
            elif piece:
                for m in PAT.finditer(piece):
                    word = m.group(0); cstart = pos + m.start()
                    b2c = [k for k, ch in enumerate(word) for _ in ch.encode("utf-8")]   # byte index -> character index
                    w = "".join(B2U[b] for b in word.encode("utf-8")); b = 0
                    for p in self._parts(w):
                        s0, e0 = cstart + b2c[b], cstart + b2c[b + len(p) - 1] + 1
                        lead = len(p) - len(p.lstrip("Ġ")); trail = 0   # Ġ is the symbol for a space byte
                        s1 = min(s0 + lead, e0)
                        ids.append(self.vocab.get(p, 0)); toks.append(p); offs.append((s1, e0)); b += len(p)
            pos += len(piece)
        e = _Enc(ids); e.tokens, e.offsets = toks, offs
        return e
    def encode_batch(self, texts):
        out = []
        for t in texts:
            e = _Enc(self.encode_ids(t)); out.append(e)      # batch use only needs ids; use encode() when offsets are needed
        return out
    def decode(self, ids):
        out = bytearray()
        for i in ids:
            t = self.id2tok[int(i)]
            out += t.encode() if t in SPECIALS else bytes(U2B[c] for c in t)
        return out.decode("utf-8", errors="replace")
    @classmethod
    def from_json(cls, path):
        d = json.load(open(path)); m = d.get("model", d)       # accepts the saved merge-table file
        return cls(m["vocab"], [tuple(x.split(" ")) if isinstance(x, str) else tuple(x) for x in m["merges"]])
    def save(self, path):
        json.dump({"vocab": self.vocab, "merges": [list(m) for m in self.merges]}, open(path, "w"))


In [5]:
import os, json, glob, shutil, torch
from google.colab import drive
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")

DRIVE_PROJECT_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
shutil.copy(f"{DRIVE_PROJECT_DIR}/legal_bpe_32k_FINAL.json", "legal_bpe_32k_FINAL.json")
tokenizer = ByteBPE.from_json("legal_bpe_32k_FINAL.json")
assert tokenizer.get_vocab_size() == 32000
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# HF_TOKEN = "..."   # set it as before (the Phase-2 setup cell used it)
print("device:", device, "| vocab:", tokenizer.get_vocab_size())
for p in sorted(glob.glob(f"{DRIVE_PROJECT_DIR}/**/*.pt", recursive=True)): print(" ", p)

device: cuda | vocab: 32000
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/alt_stride_32_128_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/best_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/encdec_6x6_checkpoints/last_debug.pt
  /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/gqa_checkpoints/gqa_best.pt
  /content/drive/MyDrive/CS781_Capstone_Proj

In [ ]:
# ==========================================
# Train the 32k byte-level BPE tokenizer with our own trainer (train_bpe, defined above)
# ==========================================
import json, time
from huggingface_hub import hf_hub_download

TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
]
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

def training_text_generator():
    doc_count = 0
    for file_path in TRAIN_FILES:
        local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=file_path, repo_type="dataset")
        with open(local_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            for record in (list(data.values()) if isinstance(data, dict) else data):
                doc_count += 1
                if doc_count % 5000 == 0: print(f"  ...streamed {doc_count} documents so far")
                if isinstance(record, dict) and "text" in record: yield record["text"]
                elif isinstance(record, str): yield record
    print(f"Finished streaming. Total documents yielded: {doc_count}")

t0 = time.time()
vocab, merges = train_bpe(training_text_generator(), vocab_size=32000, min_frequency=2)
print(f"Trained in {time.time() - t0:.0f}s | vocab size: {len(vocab)} | merges: {len(merges)}")

OUTPUT_PATH = "legal_bpe_32k_scratch.json"
ByteBPE(vocab, merges).save(OUTPUT_PATH)
print("Saved to", OUTPUT_PATH)

# ---- sanity check: round trip ----
trained = ByteBPE.from_json(OUTPUT_PATH)
for text in ["The Supreme Court held that statutory interpretation must adhere to the text.", "A", "the court held that"]:
    enc = trained.encode(text); dec = trained.decode(enc.ids)
    print(f"{len(enc.ids):3d} tokens | {enc.ids} | {dec}")
    assert dec == text, "round trip failed"

# ---- compare with the tokenizer file used everywhere downstream ----
import os
ref_path = "legal_bpe_32k_FINAL.json"
if os.path.exists(ref_path):
    ref = ByteBPE.from_json(ref_path)
    print("vocab identical :", trained.vocab == ref.vocab)
    print("merges identical:", trained.merges == ref.merges)
    first_diff = next((i for i, (a, b) in enumerate(zip(trained.merges, ref.merges)) if a != b), None)
    print("first differing merge:", first_diff)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Pick a clear destination folder - adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1"
os.makedirs(DEST_DIR, exist_ok=True)

src_path = "legal_bpe_32k_FINAL.json"
dest_path = os.path.join(DEST_DIR, "legal_bpe_32k_FINAL.json")

shutil.copy(src_path, dest_path)
print(f"Saved to: {dest_path}")

# Quick confirmation it actually copied correctly
check = ByteBPE.from_json(dest_path)
print(f"Verified — loads fine from Drive, vocab size: {check.get_vocab_size()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved to: /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/legal_bpe_32k_FINAL.json
Verified — loads fine from Drive, vocab size: 32000


## 5. Download Training Data

In [ ]:
from huggingface_hub import hf_hub_download

train_file_paths = []

for file_path in TRAIN_FILES:
    print(f"Downloading '{file_path}' from repository '{HF_DATASET_REPO}'...")
    downloaded_path = hf_hub_download(
        repo_id=HF_DATASET_REPO,
        filename=file_path,
        repo_type="dataset",
        token=HF_TOKEN
    )
    train_file_paths.append(downloaded_path)
    print(f"✅ Downloaded to: {downloaded_path}\n")

print("All selected training files downloaded successfully!")
print(train_file_paths)

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json

✅ Downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json

All selected training files downloaded successfully!
['/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_acts.json', '/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json']


## 6. Create Train and Validation Tokens Set

In [ ]:
import json
from huggingface_hub import hf_hub_download
from datasets import Dataset
import numpy as np
import os
import time
from tqdm import tqdm

# 1. Load your verified tokenizer — MAKE SURE THIS IS THE CORRECT FILE
tokenizer_path = "legal_bpe_32k_FINAL.json"
tokenizer = ByteBPE.from_json(tokenizer_path)
print(f"✅ Loaded tokenizer from {tokenizer_path} (Vocab size: {tokenizer.get_vocab_size()})")

_test = tokenizer.encode("the court held that")
assert tokenizer.decode(_test.ids).strip() != "", "Tokenizer decode failed sanity check — STOP."
print("Tokenizer sanity check passed.")

# 2. Gather raw text — everything except high courts
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILES = [
    "Phase_1_Pretraining/Train/train_acts.json",
    "Phase_1_Pretraining/Train/train_supreme_court.json",
]

all_texts = []
for file_path in tqdm(TRAIN_FILES, desc="Downloading & Reading Dataset Files"):
    local_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=file_path, repo_type="dataset")
    with open(local_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        records = list(data.values()) if isinstance(data, dict) else data
        for record in records:
            text = record.get("text") if isinstance(record, dict) else record
            if text and isinstance(text, str):
                all_texts.append(text)

print(f"Total raw texts collected: {len(all_texts)}")

# 3. Train/Val split
raw_dataset = Dataset.from_dict({"text": all_texts})
split_dataset = raw_dataset.train_test_split(test_size=0.05, seed=42)
print(f"Train samples: {len(split_dataset['train'])} | Val samples: {len(split_dataset['test'])}")

# 4. Tokenize + pack — SMALL batch_size and writer_batch_size to avoid memory pressure/swapping
BLOCK_SIZE = 2048
endoftext_id = tokenizer.token_to_id("<|endoftext|>")
if endoftext_id is None:
    endoftext_id = 256

def tokenize_and_pack(examples):
    batch_encodings = tokenizer.encode_batch(examples["text"])
    all_token_ids = []
    for encoding in batch_encodings:
        all_token_ids.extend(encoding.ids)
        all_token_ids.append(endoftext_id)

    total_length = (len(all_token_ids) // BLOCK_SIZE) * BLOCK_SIZE
    all_token_ids = all_token_ids[:total_length]
    chunked_ids = np.array(all_token_ids, dtype=np.int32).reshape(-1, BLOCK_SIZE)

    return {"input_ids": chunked_ids.tolist(), "labels": chunked_ids.tolist()}

PACK_BATCH_SIZE = 500      # small, avoids huge in-RAM lists per batch
WRITER_BATCH_SIZE = 500    # flush to disk often, don't hoard memory
NUM_PROC = 2                # safe default for Colab; don't blindly trust os.cpu_count()

print(f"Detected CPU count: {os.cpu_count()} (using NUM_PROC={NUM_PROC} regardless)")

print(f"⚙️ Tokenizing and packing training set...")
t0 = time.time()
tokenized_train = split_dataset["train"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Train Split"
)
print(f"Train packing took {time.time() - t0:.1f}s")

print("⚙️ Tokenizing and packing validation set...")
t0 = time.time()
tokenized_val = split_dataset["test"].map(
    tokenize_and_pack,
    batched=True,
    batch_size=PACK_BATCH_SIZE,
    num_proc=NUM_PROC,
    writer_batch_size=WRITER_BATCH_SIZE,
    remove_columns=["text"],
    desc="Processing Val Split"
)
print(f"Val packing took {time.time() - t0:.1f}s")

# 5. Save
output_dir = "packed_pretrain_data"
os.makedirs(output_dir, exist_ok=True)
tokenized_train.save_to_disk(os.path.join(output_dir, "train"))
tokenized_val.save_to_disk(os.path.join(output_dir, "val"))

print(f"\n🎉 Saved to '{output_dir}/'")
print(f"   - Train blocks: {len(tokenized_train)} (~{len(tokenized_train) * BLOCK_SIZE / 1e6:.2f}M tokens)")
print(f"   - Val blocks:   {len(tokenized_val)} (~{len(tokenized_val) * BLOCK_SIZE / 1e6:.2f}M tokens)")

# 6. Verify packed data decodes correctly
sample_ids = tokenized_train[0]["input_ids"]
decoded_sample = tokenizer.decode(sample_ids[:50])
print(f"\n--- Packed block decode check ---\n{decoded_sample}")
assert decoded_sample.strip() != "", "Packed data decodes to empty — tokenizer/data mismatch!"
print("Packed data verified readable.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
import os

# Destination folder on Drive — adjust if you want a different location
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/packed_pretrain_data"
os.makedirs(DEST_DIR, exist_ok=True)

src_dir = "packed_pretrain_data"  # the local folder your script already saved to

for split in ["train", "val"]:
    src_path = os.path.join(src_dir, split)
    dest_path = os.path.join(DEST_DIR, split)

    if os.path.exists(dest_path):
        shutil.rmtree(dest_path)  # avoid stale leftovers from a previous run

    shutil.copytree(src_path, dest_path)
    print(f"Copied {src_path} -> {dest_path}")

print("\nDone. Verifying by reloading from Drive...")

# Verify both splits reload correctly straight from Drive
from datasets import load_from_disk

train_check = load_from_disk(os.path.join(DEST_DIR, "train"))
val_check = load_from_disk(os.path.join(DEST_DIR, "val"))

print(f"Train blocks on Drive: {len(train_check)}")
print(f"Val blocks on Drive:   {len(val_check)}")
print(f"Sample input_ids length: {len(train_check[0]['input_ids'])}")

Mounted at /content/drive


FileNotFoundError: [Errno 2] No such file or directory: 'packed_pretrain_data/train'

## 7. Create Token Blocks

In [ ]:
from datasets import load_from_disk

# Load your pre-packed training and validation blocks from disk
train_blocks = load_from_disk("packed_pretrain_data/train")
val_blocks = load_from_disk("packed_pretrain_data/val")

print(f"✅ Loaded train_blocks: {len(train_blocks):,} sequences")
print(f"✅ Loaded val_blocks:   {len(val_blocks):,} sequences")

✅ Loaded train_blocks: 190,794 sequences
✅ Loaded val_blocks:   10,207 sequences





## 8. Create Datasets

In [ ]:
# train_blocks and val_blocks are already Hugging Face Datasets!
print(train_blocks)
print(val_blocks)

# You can inspect columns directly:
print(train_blocks[0])

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 190794
})
Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 10207
})
{'input_ids': [1443, 484, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 5512, 2173, 1934, 2900, 1753, 13, 235, 1589, 492, 2899, 13, 3714, 4064, 241, 2073, 2183, 2368, 308, 19616, 1703, 10, 372, 15, 280, 15, 342, 15, 24340, 308, 38, 922, 655, 325, 1050, 1616, 521, 292, 235, 2266, 294, 2401, 295, 1672, 1633, 25828, 36, 15, 395, 15, 14177, 3, 1443, 484, 4385, 3520, 308, 9017, 19669, 241, 2632, 1318, 5576, 14, 11654, 10823, 10, 1616, 13, 3714, 440, 1365, 241, 235, 1301, 2226, 292, 1110, 308, 18, 10, 241, 2749, 14869, 43, 241, 235, 991, 13, 235, 2266, 2458, 3056, 13, 343, 803, 259, 235, 484, 241, 3520, 13, 235, 1050, 1616, 13, 1618, 3549, 180, 18, 15, 4477, 2312, 268, 2083, 15, 180, 9, 18, 10, 948, 1616, 414, 274, 1847, 235, 1008, 241, 3520, 308, 9017, 19669, 241, 2632, 295, 5576, 14, 11654, 10823, 10, 161

## 9. Define The Model

In [6]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. CORE BUILDING BLOCKS & MLA
# ==========================================
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(variance + self.eps) * self.weight


def precompute_rope_params(head_dim: int, max_seq_len: int = 4096, theta: float = 10000.0):
    assert head_dim % 2 == 0, "Head dimension must be even for RoPE"
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(max_seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs_cos = torch.cos(freqs).repeat_interleave(2, dim=-1)
    freqs_sin = torch.sin(freqs).repeat_interleave(2, dim=-1)
    return freqs_cos, freqs_sin


def apply_rotary_emb(x, cos, sin):
    def rotate_half(tensor):
        x1 = tensor[..., 0::2]
        x2 = tensor[..., 1::2]
        return torch.stack([-x2, x1], dim=-1).flatten(-2)
    cos = cos.unsqueeze(0).unsqueeze(1)
    sin = sin.unsqueeze(0).unsqueeze(1)
    return (x * cos) + (rotate_half(x) * sin)


class SwiGLU(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        hidden_dim = int(2.5 * d_model)
        hidden_dim = ((hidden_dim + 255) // 256) * 256
        self.w1 = nn.Linear(d_model, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, d_model, bias=False)
        self.w3 = nn.Linear(d_model, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model: int, n_head: int, max_seq_len: int = 4096,
                 kv_lora_rank: int = 128, q_lora_rank: int = 256, qk_rope_head_dim: int = 64):
        super().__init__()
        self.d_model = d_model
        self.n_head = n_head
        self.head_dim = d_model // n_head
        self.kv_lora_rank = kv_lora_rank
        self.qk_rope_head_dim = qk_rope_head_dim

        self.kv_down_proj = nn.Linear(d_model, kv_lora_rank + qk_rope_head_dim, bias=False)
        self.kv_norm = RMSNorm(kv_lora_rank)
        self.kv_up_proj = nn.Linear(kv_lora_rank, n_head * (self.head_dim * 2), bias=False)

        self.q_down_proj = nn.Sequential(nn.Linear(d_model, q_lora_rank, bias=False), RMSNorm(q_lora_rank))
        self.q_up_proj = nn.Linear(q_lora_rank, n_head * (self.head_dim + qk_rope_head_dim), bias=False)

        self.out_proj = nn.Linear(n_head * self.head_dim, d_model, bias=False)

        mask = torch.full((max_seq_len, max_seq_len), float("-inf"))
        mask = torch.triu(mask, diagonal=1)
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x, cos, sin, past_kv=None):
        B, T, C = x.shape

        q_latent = self.q_down_proj(x)
        q_projected = self.q_up_proj(q_latent).view(B, T, self.n_head, self.head_dim + self.qk_rope_head_dim)
        q_nope, q_rope = torch.split(q_projected, [self.head_dim, self.qk_rope_head_dim], dim=-1)

        compressed_kv = self.kv_down_proj(x)
        c_kv = compressed_kv[..., :self.kv_lora_rank]
        k_rope_new = compressed_kv[..., self.kv_lora_rank:]
        c_kv_normalized = self.kv_norm(c_kv)

        past_len = past_kv[0].shape[1] if past_kv is not None else 0
        total_len = past_len + T

        cos_sliced = cos[past_len:total_len, :]
        sin_sliced = sin[past_len:total_len, :]

        k_rope_new = k_rope_new.unsqueeze(1).expand(B, self.n_head, T, self.qk_rope_head_dim)
        q_rope = apply_rotary_emb(q_rope.transpose(1, 2), cos_sliced, sin_sliced)
        k_rope_new = apply_rotary_emb(k_rope_new, cos_sliced, sin_sliced)

        if past_kv is not None:
            c_kv_prev, k_rope_prev = past_kv
            c_kv_cache = torch.cat([c_kv_prev, c_kv_normalized], dim=1)
            k_rope_cache = torch.cat([k_rope_prev, k_rope_new], dim=2)
        else:
            c_kv_cache = c_kv_normalized
            k_rope_cache = k_rope_new

        present_kv = (c_kv_cache, k_rope_cache)

        kv_expanded = self.kv_up_proj(c_kv_cache).view(B, -1, self.n_head, 2 * self.head_dim)
        k_nope_cache, v_cache = torch.split(kv_expanded, self.head_dim, dim=-1)

        k_nope_cache = k_nope_cache.transpose(1, 2)
        v_cache = v_cache.transpose(1, 2)
        q_nope = q_nope.transpose(1, 2)

        q_full = torch.cat([q_nope, q_rope], dim=-1)
        k_full = torch.cat([k_nope_cache, k_rope_cache], dim=-1)

        scores = torch.matmul(q_full, k_full.transpose(-2, -1)) / math.sqrt(self.head_dim + self.qk_rope_head_dim)
        scores = scores + self.causal_mask[past_len:total_len, :total_len]

        attn_weights = F.softmax(scores, dim=-1)
        y = torch.matmul(attn_weights, v_cache)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(y), present_kv


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_head: int, max_seq_len: int = 4096, qk_rope_head_dim: int = 64):
        super().__init__()
        self.attention_norm = RMSNorm(d_model)
        self.attn = MultiHeadLatentAttention(d_model, n_head, max_seq_len, qk_rope_head_dim=qk_rope_head_dim)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model)

    def forward(self, x, cos, sin, past_kv=None):
        attn_out, present_kv = self.attn(self.attention_norm(x), cos, sin, past_kv=past_kv)
        x = x + attn_out
        x = x + self.ffn(self.ffn_norm(x))
        return x, present_kv


# ==========================================
# 2. MODEL WRAPPER & INITIALIZATION
# ==========================================
class CustomDecoderLM(nn.Module):
    """Full Decoder-Only Language Model from Scratch with MLA + KV Caching."""
    def __init__(self, vocab_size=32000, d_model=1024, n_layer=12, n_head=16,
                 max_seq_len=4096, qk_rope_head_dim=64, tie_weights=True):
        super().__init__()
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len

        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_head, max_seq_len, qk_rope_head_dim=qk_rope_head_dim)
            for _ in range(n_layer)
        ])
        self.norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        if tie_weights:
            self.token_embedding.weight = self.lm_head.weight

        # RoPE dimension MUST match qk_rope_head_dim, not head_dim (d_model // n_head)
        cos, sin = precompute_rope_params(qk_rope_head_dim, max_seq_len)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, input_ids, labels=None, past_key_values=None):
        B, T = input_ids.shape
        x = self.token_embedding(input_ids)

        new_present_key_values = []
        for idx, layer in enumerate(self.layers):
            layer_past = past_key_values[idx] if past_key_values is not None else None
            x, present_kv = layer(x, self.cos, self.sin, past_kv=layer_past)
            new_present_key_values.append(present_kv)

        x = self.norm(x)
        logits = self.lm_head(x)

        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, self.vocab_size), shift_labels.view(-1))

        return {
            "loss": loss,
            "logits": logits,
            "past_key_values": tuple(new_present_key_values)
        }


# ==========================================
# 3. DETERMINE VOCAB SIZE — explicit, no silent fallback
# ==========================================
assert 'tokenizer' in globals(), "Load your verified tokenizer (e.g. legal_bpe_32k_FINAL.json) before building the model."
vocab_sz = tokenizer.get_vocab_size()
print(f"Using tokenizer vocab size: {vocab_sz}")

target_device = DEVICE if 'DEVICE' in globals() else (device if 'device' in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# ==========================================
# 4. BUILD MODEL — scaled down to ~90M parameters
# ==========================================
model = CustomDecoderLM(
    vocab_size=vocab_sz,
    d_model=768,
    n_layer=10,
    n_head=12,
    max_seq_len=4096,
    qk_rope_head_dim=64,
    tie_weights=True
).to(target_device)

model_size = sum(p.numel() for p in model.parameters())
print(f"🚀 Custom MLA Decoder Model Size: {model_size / 1e6:.2f}M parameters")

Using tokenizer vocab size: 32000
🚀 Custom MLA Decoder Model Size: 87.02M parameters


## 10. Training Arguments

In [ ]:
from datasets import load_from_disk

# 1. Load the pre-packed datasets from disk
train_dataset = load_from_disk("packed_pretrain_data/train")
val_dataset = load_from_disk("packed_pretrain_data/val")

# 2. Set format to PyTorch tensors
train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

Loading dataset from disk:   0%|          | 0/44 [00:00<?, ?it/s]

In [ ]:
from datasets import load_from_disk
import torch

DATA_DIR = "packed_pretrain_data"  # or your Drive path if this is a fresh session
train_dataset = load_from_disk(f"{DATA_DIR}/train")
val_dataset = load_from_disk(f"{DATA_DIR}/val")

train_dataset.set_format(type="torch", columns=["input_ids", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "labels"])

print(f"Train blocks: {len(train_dataset)} | Val blocks: {len(val_dataset)}")

Train blocks: 190794 | Val blocks: 10207


## 11. Training

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [ ]:
import math
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 2
    MAX_STEPS_PER_EPOCH = 20
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 5
    COOLDOWN_STEPS = 5
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 3
    BATCH_SIZE = 2
    MAX_STEPS_PER_EPOCH = 10000
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 500
    COOLDOWN_STEPS = 1000

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 2. Dataloaders
# ============================================================
def custom_collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

train_subset_size = min(len(train_dataset), MAX_STEPS_PER_EPOCH * BATCH_SIZE)
train_subset = Subset(train_dataset, range(train_subset_size))
val_subset_size = min(len(val_dataset), 500 * BATCH_SIZE)
val_subset = Subset(val_dataset, range(val_subset_size))

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custom_collate_fn)
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}")

# ============================================================
# 3. Optimizer & schedule
# ============================================================
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
total_steps = MAX_STEPS_PER_EPOCH * EPOCHS

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 4. Evaluation
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

# ============================================================
# 5. Training loop
# ============================================================
model.train()
val_loss_history = []

print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

for epoch in range(EPOCHS):
    progress_bar = tqdm(train_loader, total=min(len(train_loader), MAX_STEPS_PER_EPOCH), desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad()
    epoch_train_loss = torch.tensor(0.0, device=device)
    valid_steps = 0

    for step, batch in enumerate(progress_bar):
        if step >= MAX_STEPS_PER_EPOCH:
            break
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            current_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS
            epoch_train_loss += current_loss
            valid_steps += 1

            progress_bar.set_postfix({
                "loss": f"{current_loss.item():.4f}",
                "val_loss": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                "lr": f"{scheduler.get_last_lr()[0]:.6f}"
            })

    avg_train_loss = (epoch_train_loss / max(1, valid_steps)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")

print("✅ Training complete!")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

Train batches/epoch: 10000 | Val batches: 500
Using bf16: True

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...


Epoch 1/3:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 1 -> Train Loss: 3.9790 | Val Loss: 3.3347



Epoch 2/3:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 2 -> Train Loss: 3.0020 | Val Loss: 2.9438



Epoch 3/3:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 3 -> Train Loss: 2.7523 | Val Loss: 2.8261

✅ Training complete!


In [ ]:
import math
import torch
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

# ============================================================
# 0. Guard — fail fast with a clear message if something's missing
# ============================================================
for name in ["tokenizer", "model", "train_dataset", "val_dataset", "device"]:
    assert name in globals(), f"'{name}' is not defined — run your earlier setup cells first."

vocab_sz = tokenizer.get_vocab_size()
assert model.vocab_size == vocab_sz, f"model.vocab_size={model.vocab_size} != tokenizer vocab={vocab_sz}"

# ============================================================
# 1. Toggle: smoke-test first, then flip to False for the real run
# ============================================================
DEBUG_MODE = False

if DEBUG_MODE:
    EPOCHS = 1
    BATCH_SIZE = 2
    MAX_STEPS_PER_EPOCH = 20
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 5
    COOLDOWN_STEPS = 5
    print("*** DEBUG_MODE is ON — tiny smoke test only ***\n")
else:
    EPOCHS = 5
    BATCH_SIZE = 2
    MAX_STEPS_PER_EPOCH = 10000
    GRADIENT_ACCUMULATION_STEPS = 2
    WARMUP_STEPS = 500
    COOLDOWN_STEPS = 1000

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0

# ============================================================
# 2. Dataloaders
# ============================================================
def custom_collate_fn(batch):
    return {
        "input_ids": torch.stack([item["input_ids"] for item in batch]),
        "labels": torch.stack([item["labels"] for item in batch]),
    }

train_subset_size = min(len(train_dataset), MAX_STEPS_PER_EPOCH * BATCH_SIZE)
train_subset = Subset(train_dataset, range(train_subset_size))
val_subset_size = min(len(val_dataset), 500 * BATCH_SIZE)
val_subset = Subset(val_dataset, range(val_subset_size))

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custom_collate_fn)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=custom_collate_fn)
print(f"Train batches/epoch: {len(train_loader)} | Val batches: {len(val_loader)}")

# ============================================================
# 3. Optimizer & schedule
# ============================================================
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95))
total_steps = MAX_STEPS_PER_EPOCH * EPOCHS

def get_lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return float(current_step) / float(max(1, WARMUP_STEPS))
    elif current_step > (total_steps - COOLDOWN_STEPS):
        progress = float(total_steps - current_step) / float(max(1, COOLDOWN_STEPS))
        return max(0.1, progress)
    return 1.0

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_lambda)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"Using bf16: {use_bf16}")

model.to(device)

# ============================================================
# 4. Evaluation
# ============================================================
@torch.no_grad()
def evaluate():
    model.eval()
    total_val_loss = torch.tensor(0.0, device=device)
    n_batches = 0
    for batch in val_loader:
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            total_val_loss += outputs["loss"].detach()
        n_batches += 1
    model.train()
    return (total_val_loss / max(1, n_batches)).item()

# ============================================================
# 5. Training loop
# ============================================================
model.train()
val_loss_history = []

print(f"\nExpected initial loss ≈ ln({vocab_sz}) ≈ {math.log(vocab_sz):.2f} (untrained baseline)\n")
print("🚀 Starting Training Loop...")

for epoch in range(EPOCHS):
    progress_bar = tqdm(train_loader, total=min(len(train_loader), MAX_STEPS_PER_EPOCH), desc=f"Epoch {epoch+1}/{EPOCHS}")
    optimizer.zero_grad()
    epoch_train_loss = torch.tensor(0.0, device=device)
    valid_steps = 0

    for step, batch in enumerate(progress_bar):
        if step >= MAX_STEPS_PER_EPOCH:
            break
        input_ids, labels = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16 if use_bf16 else torch.float16):
            outputs = model(input_ids=input_ids, labels=labels)
            loss = outputs["loss"] / GRADIENT_ACCUMULATION_STEPS

        loss.backward()

        if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            current_loss = loss.detach() * GRADIENT_ACCUMULATION_STEPS
            epoch_train_loss += current_loss
            valid_steps += 1

            progress_bar.set_postfix({
                "loss": f"{current_loss.item():.4f}",
                "val_loss": f"{val_loss_history[-1]:.4f}" if val_loss_history else "N/A",
                "lr": f"{scheduler.get_last_lr()[0]:.6f}"
            })

    avg_train_loss = (epoch_train_loss / max(1, valid_steps)).item()
    avg_val_loss = evaluate()
    val_loss_history.append(avg_val_loss)
    print(f"\n📊 Epoch {epoch+1} -> Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}\n")

print("✅ Training complete!")
if DEBUG_MODE:
    print("\n*** DEBUG_MODE smoke test finished. If loss looked sane, set DEBUG_MODE=False and re-run. ***")

Train batches/epoch: 10000 | Val batches: 500
Using bf16: True

Expected initial loss ≈ ln(32000) ≈ 10.37 (untrained baseline)

🚀 Starting Training Loop...


Epoch 1/5:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 1 -> Train Loss: 3.9728 | Val Loss: 3.2456



Epoch 2/5:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 2 -> Train Loss: 2.9603 | Val Loss: 2.9300



Epoch 3/5:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 3 -> Train Loss: 2.7441 | Val Loss: 2.8180



Epoch 4/5:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 4 -> Train Loss: 2.6174 | Val Loss: 2.7609



Epoch 5/5:   0%|          | 0/10000 [00:00<?, ?it/s]


📊 Epoch 5 -> Train Loss: 2.5377 | Val Loss: 2.7269

✅ Training complete!


In [ ]:
import os
import torch

# ============================================================
# Save model (and optionally optimizer/scheduler state for resuming)
# ============================================================
SAVE_DIR = "model_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(SAVE_DIR, "custom_decoder_lm.pt")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "vocab_size": model.vocab_size,
    "d_model": 768,
    "n_layer": 10,
    "n_head": 12,
    "max_seq_len": model.max_seq_len,
    "val_loss_history": val_loss_history,
}, CHECKPOINT_PATH)

print(f"Saved checkpoint to {CHECKPOINT_PATH}")

# ============================================================
# Verify it loads back correctly before trusting it
# ============================================================
check = torch.load(CHECKPOINT_PATH, map_location=device)
print(f"Checkpoint contains keys: {list(check.keys())}")
print(f"Model vocab_size in checkpoint: {check['vocab_size']}")
assert check['vocab_size'] == tokenizer.get_vocab_size(), "Mismatch between saved checkpoint and current tokenizer!"
print("✅ Checkpoint verified loadable and consistent with current tokenizer.")

Saved checkpoint to model_checkpoints/custom_decoder_lm.pt
Checkpoint contains keys: ['model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'vocab_size', 'd_model', 'n_layer', 'n_head', 'max_seq_len', 'val_loss_history']
Model vocab_size in checkpoint: 32000
✅ Checkpoint verified loadable and consistent with current tokenizer.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
DEST_DIR = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/model_checkpoints"
os.makedirs(DEST_DIR, exist_ok=True)
shutil.copy(CHECKPOINT_PATH, os.path.join(DEST_DIR, "custom_decoder_lm.pt"))
print(f"Copied checkpoint to {DEST_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied checkpoint to /content/drive/MyDrive/CS781_Capstone_Project_Phase_1/model_checkpoints


In [ ]:
checkpoint = torch.load("model_checkpoints/custom_decoder_lm.pt", map_location=device)

model = CustomDecoderLM(
    vocab_size=checkpoint["vocab_size"],
    d_model=checkpoint["d_model"],
    n_layer=checkpoint["n_layer"],
    n_head=checkpoint["n_head"],
    max_seq_len=checkpoint["max_seq_len"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
print("Model loaded and ready for inference.")

Model loaded and ready for inference.


## 12. Perplexity

In [ ]:
import math

# After your training loop finishes:
final_val_loss = val_loss_history[-1] # Grabs the last epoch's validation loss
perplexity = math.exp(final_val_loss)

print(f"📊 Final Validation Loss: {final_val_loss:.6f}")
print(f"📈 Final Validation Perplexity: {perplexity:.4f}")

📊 Final Validation Loss: 2.826090
📈 Final Validation Perplexity: 16.8793


## 13. Plot Training and Validation Loss

## 16. Load Test Data

In [ ]:
from huggingface_hub import hf_hub_download

# Make sure filename points to the correct nested directory path
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file downloaded to:", test_file_path)

Test file downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json


## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

Number of test examples: 1000


## 18. Load Final Model for Generation

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F

# --- 1. Find and Load Model Weights ---
model_files = glob.glob("**/pytorch_model.bin", recursive=True)
if not model_files:
    model_files = glob.glob("/content/**/*.bin", recursive=True)

if not model_files:
    raise FileNotFoundError("Could not find 'pytorch_model.bin' weights file in the VM.")

state_dict_path = model_files[0]
print(f"✅ Found model weights at: {state_dict_path}")

# --- 2. Instantiate with Checkpoint's Vocab Size (50257) ---
generation_model = CustomDecoderLM(
    vocab_size=50257,  # Matches the checkpoint shape from training
    d_model=768,
    n_layer=8,
    n_head=12,
    max_seq_len=1024
)

generation_model.load_state_dict(torch.load(state_dict_path, map_location="cpu"))

# --- 3. Move to GPU & Eval Mode ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device)
generation_model.eval()

print(f"🚀 Device: {device}")
print(f"📊 Model Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

✅ Found model weights at: drive/MyDrive/CS781_Capstone_Project_Phase_1/pytorch_model.bin
🚀 Device: cuda
📊 Model Parameters: 95.23M


In [ ]:
# Load your saved weights before running generation
checkpoint_path = "/content/drive/MyDrive/CS781_Capstone_Project_Phase_1/DeepSeek/deepseek_mla_85m_pretrained.bin"

if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()
    print("✅ Successfully loaded pretrained model weights from Google Drive!")
else:
    print("⚠️ Warning: Checkpoint path not found. Model is currently using random initialization weights.")

✅ Successfully loaded pretrained model weights from Google Drive!


## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
import json
import os
import torch
from tqdm.auto import tqdm

# ============================================================
# 0. Sanity checks — same discipline as the rest of this project
# ============================================================
for name in ["tokenizer", "model", "device"]:
    assert name in globals(), f"'{name}' is not defined — load/build it before running this cell."

model.eval()

# quick decode check before generating anything real
_t = tokenizer.encode("the court held that")
assert tokenizer.decode(_t.ids).strip() != "", "Tokenizer sanity check failed — stop and check tokenizer."

# ============================================================
# 1. Load test data
# ============================================================
from huggingface_hub import hf_hub_download

TEST_FILE = "Phase_1_Pretraining/Test/test.json"

test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)
print("Test file downloaded to:", test_file_path)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
print("Number of test examples:", len(test_data))

# ============================================================
# 2. Generation function — word-count based stopping, KV cache for speed
# ============================================================
TARGET_WORDS = 128
MAX_PROMPT_TOKENS = 1024   # safety cap; model's max_seq_len is 4096, plenty of headroom
MAX_NEW_TOKENS = int(TARGET_WORDS * 2.2)  # generous token budget to reach 128 real words

endoftext_id = tokenizer.token_to_id("<|endoftext|>")
if endoftext_id is None:
    endoftext_id = 256

@torch.no_grad()
def generate_continuation(prompt_text, target_words=TARGET_WORDS):
    enc = tokenizer.encode(prompt_text)
    prompt_ids = enc.ids[:MAX_PROMPT_TOKENS]  # truncate overly long prompts defensively

    input_ids = torch.tensor([prompt_ids], dtype=torch.long, device=device)

    # 1. Prime the KV cache with the full prompt
    outputs = model(input_ids, past_key_values=None)
    logits = outputs["logits"]
    past_key_values = outputs["past_key_values"]

    next_token_logits = logits[:, -1, :]
    next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
    generated_ids = [next_token.item()]

    # 2. Autoregressive generation loop
    for _ in range(MAX_NEW_TOKENS - 1):
        outputs = model(next_token, past_key_values=past_key_values)
        logits = outputs["logits"]
        past_key_values = outputs["past_key_values"]

        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        token_id = next_token.item()
        generated_ids.append(token_id)

        if token_id == endoftext_id:
            break

        # stop early once we already have enough words — avoids wasted compute
        if len(generated_ids) % 10 == 0:
            partial_text = tokenizer.decode(generated_ids)
            if len(partial_text.split()) >= target_words:
                break

    continuation_text = tokenizer.decode(generated_ids)

    # Trim to exactly target_words (in case we overshot)
    words = continuation_text.split()
    if len(words) > target_words:
        continuation_text = " ".join(words[:target_words])

    return continuation_text

# ============================================================
# 3. Run generation over the full test set
# ============================================================
predictions = {}
items = list(test_data.items())

for example_id, record in tqdm(items, desc="Generating predictions"):
    prompt_text = record["text"] if isinstance(record, dict) else record
    try:
        continuation = generate_continuation(prompt_text, target_words=TARGET_WORDS)
    except Exception as e:
        print(f"⚠️ Generation failed for ID {example_id}: {e}")
        continuation = ""
    predictions[str(example_id)] = {"text": continuation}

# ============================================================
# 4. Verify before saving — catch empty/degenerate output NOW
# ============================================================
empty_count = sum(1 for v in predictions.values() if not v["text"].strip())
print(f"\n{empty_count} / {len(predictions)} predictions are empty strings.")
if empty_count > 0:
    print("⚠️ Some predictions are empty — check the tokenizer/model pairing before submitting.")

sample_id = items[0][0]
print(f"\n--- Sample prediction (ID {sample_id}) ---")
print(predictions[sample_id]["text"][:500])

# ============================================================
# 5. Save prediction.json
# ============================================================
output_file = "prediction.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(predictions)} predictions to '{output_file}'")

Test file downloaded to: /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Test/test.json
Number of test examples: 1000


Generating predictions:   0%|          | 0/1000 [00:00<?, ?it/s]


0 / 1000 predictions are empty strings.

--- Sample prediction (ID 1) ---
not necessary to consider the correctness of the decision of this Court in [State of Karnataka Vs. M.P. State Electricity Board], (1997) 3 SCC 784. The Court has also observed that the decision of this Court in [State of Karnataka Vs. M.P. State Electricity Board], (1997) 3 SCC 784, cannot be said to be arbitrary or arbitrary. The Court has also observed that the decision of this Court in [State of Karnataka Vs. M.P. State Electricity Board], (1997) 3 SCC 784, cannot be said to be arbitrary or a

Saved 1000 predictions to 'prediction.json'
